# Module 3.1: Memory vs RAG vs Context Engineering

Every time a user returns to an agent, the agent has **forgotten everything**.
It doesn't know Sarah prefers Marriott. It doesn't know she rated it 5/5 last time.
It doesn't know her budget constraints.

> **The question**: How do we give agents access to user-specific, evolving knowledge?

Three paradigms exist — each with different tradeoffs. This notebook demonstrates
all three on the **same query** so you can feel the difference.

In [ ]:
%pip install -q -r ../requirements.txt azure-search-documents openai

In [ ]:
import sys, os, json
import sniffio

sys.path.insert(0, "..")

# Fix for Python 3.14: nest_asyncio breaks sniffio/httpcore.
# Instead, set the async library context var so sniffio detects asyncio correctly.
sniffio.current_async_library_cvar.set("asyncio")

from shared.travel_agent import (
    create_client, SYSTEM_PROMPT,
    search_flights, search_hotels, get_travel_policy,
)

client, credential = create_client("../.env")
print("Client ready")

## The Problem: A Stateless Agent Forgets Everything

First, let's see what happens when the agent has NO memory and NO external knowledge.
Sarah asks for a hotel recommendation — the agent has nothing personal to work with.

In [ ]:
from agent_framework import Agent, AgentSession

stateless_agent = Agent(
    client=client,
    name="StatelessAssistant",
    instructions=SYSTEM_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy],
)

async def ask_stateless():
    session = AgentSession()
    query = "I need a hotel in New York for next week. What do you recommend?"
    print(f"Query: {query}\n")
    result = await stateless_agent.run(query, session=session)
    print(f"[CONTEXT-ONLY] {result.text}")

await ask_stateless()

## What Went Wrong?

The agent returns **generic** results — any hotel, any chain, no personalisation.
It doesn't know Sarah prefers Marriott, rated it 5/5 last time, or has a loyalty program.
Every interaction starts from zero. This is the world without memory.

| What the agent said | What Sarah wanted |
|---------------------|-------------------|
| Generic hotel list from search | "Marriott — I stayed there last time and loved it" |
| No price awareness | "Keep it under $250/night, that's my budget" |
| No loyalty mention | "I have Marriott Bonvoy, use my points" |

## Partial Fix: RAG (Static Knowledge via AI Search)

RAG gives the agent access to **corporate policies** — budget limits, vendor preferences,
safety rules. But this knowledge is the SAME for everyone. It can answer "what's the
budget limit?" but NOT "what hotel does Sarah specifically prefer?"

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.models import VectorizedQuery
from azure.core.credentials import AzureKeyCredential
from openai import OpenAI
from azure.identity import get_bearer_token_provider
from agent_framework import tool

SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"]
SEARCH_KEY = os.environ["AZURE_SEARCH_KEY"]
FOUNDRY_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
EMBEDDING_MODEL = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
INDEX_NAME = "travel-policies"

# AI Search client (admin key for data-plane access)
search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY),
)

# Embedding client — uses Foundry account-level endpoint (project-level 404s for embeddings)
account_endpoint = FOUNDRY_ENDPOINT.split("/api/projects")[0]
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

openai_client = OpenAI(
    base_url=f"{account_endpoint}/openai/v1",
    api_key="placeholder",
    default_headers={"Authorization": f"Bearer {token_provider()}"},
)


def get_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(input=text[:8000], model=EMBEDDING_MODEL)
    return response.data[0].embedding


@tool
async def search_travel_policies(
    query: str, category: str = "", top_k: int = 3
) -> str:
    """Search corporate travel policies, procedures, and compliance documents.

    Use this to find current policy information about budgets, vendors,
    safety requirements, expense rules, and booking procedures.

    Args:
        query: Natural language question about travel policies
        category: Optional filter — 'policy', 'procedure', or 'compliance'
        top_k: Number of results to return (default 3)
    """
    vector_query = VectorizedQuery(
        vector=get_embedding(query),
        k_nearest_neighbors=top_k,
        fields="content_vector",
    )
    filter_expr = f"category eq '{category}'" if category else None

    results = search_client.search(
        search_text=query,
        vector_queries=[vector_query],
        filter=filter_expr,
        top=top_k,
        select=["id", "title", "content", "category", "version"],
    )
    docs = [
        {"title": r["title"], "content": r["content"][:1000], "version": r["version"]}
        for r in results
    ]
    if not docs:
        return "No matching policies found."
    return json.dumps(docs, indent=2)


# Build a RAG-only agent
rag_agent = Agent(
    client=client,
    name="RAGAssistant",
    instructions=SYSTEM_PROMPT + "\n\nYou have access to corporate travel policies via AI Search. "
        "Use the search_travel_policies tool to look up budgets, vendor preferences, "
        "safety rules, and booking procedures. Answer based on policy documents only.",
    tools=[search_flights, search_hotels, search_travel_policies],
)

print(f"RAG agent ready (index: {INDEX_NAME})")


async def ask_with_rag():
    session = AgentSession()
    query = "What's the hotel budget limit for a senior IC traveling to New York?"
    print(f"Query: {query}\n")
    result = await rag_agent.run(query, session=session)
    print(f"[RAG/AI SEARCH] {result.text}")

await ask_with_rag()

## What's Still Missing?

RAG gives the right **factual** answer from policy docs. But it cannot personalise.
Ask "what hotel does Sarah prefer?" and RAG has no answer — that's not in any
policy document. Personal preferences live in **memory**, not in a knowledge base.

| RAG can answer | RAG cannot answer |
|----------------|-------------------|
| "What's the budget limit for IC5?" | "What hotel does Sarah prefer?" |
| "Which vendors are preferred?" | "What did Sarah rate 5/5 last trip?" |
| "What's the visa rule for Japan?" | "Does Sarah have any loyalty programs?" |

## The Solution: Memory (Personalised, Evolving)

Now we give the agent access to Sarah's **episodic memory** — her past trips,
preferences, and feedback stored in Cosmos DB. Watch how the response changes.

In [ ]:
from azure.cosmos.aio import CosmosClient

COSMOS_ENDPOINT = os.environ["COSMOS_ENDPOINT"]
cosmos = CosmosClient(COSMOS_ENDPOINT, credential=credential)
db = cosmos.get_database_client("travel-memory")
episodic_container = db.get_container_client("episodic-events")
print(f"Connected to episodic memory: {COSMOS_ENDPOINT}")

In [ ]:
from agent_framework import tool

@tool
async def recall_events(
    user_id: str, event_type: str = "", limit: int = 5
) -> str:
    """Recall past events for a user. Optionally filter by event_type (trip|preference|feedback)."""
    query = "SELECT * FROM c WHERE c.user_id = @uid"
    params = [{"name": "@uid", "value": user_id}]
    if event_type:
        query += " AND c.event_type = @etype"
        params.append({"name": "@etype", "value": event_type})
    query += " ORDER BY c.timestamp DESC OFFSET 0 LIMIT @lim"
    params.append({"name": "@lim", "value": limit})

    items = [item async for item in episodic_container.query_items(
        query, parameters=params, partition_key=user_id
    )]
    if not items:
        return f"No events found for {user_id}"
    return json.dumps(items, indent=2, default=str)

memory_agent = Agent(
    client=client,
    name="MemoryAssistant",
    instructions=SYSTEM_PROMPT + "\n\nYou have access to the user's episodic memory. "
        "Before making recommendations, recall their past events and preferences. "
        "The current user is Sarah Chen (employee E001).",
    tools=[search_flights, search_hotels, get_travel_policy, recall_events],
)
print("Memory agent ready")

In [ ]:
async def ask_with_memory():
    session = AgentSession()
    query = "I need a hotel in New York for next week. What do you recommend?"
    print(f"Query: {query}\n")
    result = await memory_agent.run(query, session=session)
    print(f"[MEMORY] {result.text}")

await ask_with_memory()

## The Payoff

The memory-enabled agent recalls Sarah's past Marriott feedback and proactively
recommends it. Same query as the stateless agent above — dramatically different
(and better) response. This is **personalisation** that neither context-only
nor RAG can provide.

## Key Takeaways

| Question | Best Source | Why |
|----------|------------|-----|
| "What hotel does Sarah prefer?" | **Memory** | Learned from her past interactions |
| "What's the budget limit for IC5?" | **RAG/Policy** | Static corporate knowledge |
| "Summarise what we just discussed" | **Context** | Current session only |
| "Book the same hotel as last time" | **Memory** | Requires recall of past events |
| "What's the visa requirement for Japan?" | **RAG/Policy** | External factual knowledge |

1. **Context alone forgets everything** — each session starts from zero
2. **RAG provides facts, not personalisation** — policies are the same for everyone
3. **Memory fills the gap** — user-specific, evolving knowledge that persists across sessions
4. **All three work together** — context (session), RAG (facts), memory (personal)

## But Memory Creates New Problems

Unlike RAG (which just needs re-indexing) and context (which is ephemeral),
memory introduces challenges that require active management:

1. **What to store?** — Not everything deserves to be remembered (→ Notebook 02)
2. **When to trust?** — New memories might be wrong (→ Notebook 03: Staged Promotion)
3. **How to update?** — Facts change over time (→ Notebook 04: Belief Revision)
4. **When to forget?** — Unbounded memory degrades quality (→ Notebook 05: Retention & Decay)

The rest of this module builds each of these lifecycle capabilities.